In [2]:
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix
)

df = pd.read_csv("../data/ai4i2020.csv")

# Define data
features = [
    "Type",
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
]

target = "Machine failure"

X = df[features]
y = df[target]

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Verify
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

Machine failure
0    0.966125
1    0.033875
Name: proportion, dtype: float64
Machine failure
0    0.966
1    0.034
Name: proportion, dtype: float64


In [4]:
# Preprocessing
numeric_features = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
]

categorical_features = ["Type"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

# Create model
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000))
    ]
)

# Train model
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Verify
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[1928    4]
 [  61    7]]
              precision    recall  f1-score   support

           0       0.97      1.00      0.98      1932
           1       0.64      0.10      0.18        68

    accuracy                           0.97      2000
   macro avg       0.80      0.55      0.58      2000
weighted avg       0.96      0.97      0.96      2000



# Baseline Logistic Regression

The initial logistic regression model achieved approximately 96.75% overall
accuracy on the test set. However, this result is misleading because of the
substantial class imbalance in the dataset.

Of the 68 actual machine failures in the test set, the model correctly
identified only 7 and incorrectly classified 61 as non-failures. This
corresponds to failure-class recall of approximately 10%. Failure precision
was approximately 64%, indicating that the model's relatively few failure
predictions were often correct, but most actual failures were not detected.

This demonstrates that overall accuracy is not an appropriate primary
evaluation metric for this dataset. Failure-class precision, recall, F1 score,
and the confusion matrix provide more useful measures of model performance.

In [5]:
# Create model
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
                            max_iter=1000,
                            class_weight="balanced"
                        ))
    ]
)

# Train model
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Verify
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[1593  339]
 [  12   56]]
              precision    recall  f1-score   support

           0       0.99      0.82      0.90      1932
           1       0.14      0.82      0.24        68

    accuracy                           0.82      2000
   macro avg       0.57      0.82      0.57      2000
weighted avg       0.96      0.82      0.88      2000



# Proof-of-Concept Results

The proof-of-concept successfully demonstrated an end-to-end machine learning
pipeline using the AI4I dataset. Six machine and product characteristics were
used as predictive features, while the overall `Machine failure` column was
used as the target. Identifier columns and the individual failure-mode columns
were excluded from training.

An initial logistic regression model achieved approximately 96.75% accuracy,
but detected only 7 of the 68 failures in the test set, resulting in failure
recall of approximately 10%. This demonstrated that overall accuracy is
misleading because of the substantial class imbalance in the dataset.

A second logistic regression model was trained using balanced class weights.
This model detected 56 of the 68 failures, increasing failure recall to
approximately 82%. However, it also produced 339 false-positive failure
predictions, reducing failure precision to approximately 14% and overall
accuracy to approximately 82%.

These results demonstrate a significant precision-recall tradeoff. They also
establish a useful baseline for evaluating additional models. Further
development will investigate models capable of representing nonlinear
relationships and interactions between operating characteristics, such as
Random Forest, and compare their failure-class precision, recall, F1 score,
and confusion matrices against the logistic regression baseline.